In [81]:
import os
import soundfile as sf
import torch
from torch import nn    
import numpy as np
import librosa
from torch.nn import functional as F

In [82]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [83]:
class Demucs(nn.Module):
    def __init__(self,inchannels = 1, outchannels = 2, features = [32, 64, 128, 256]):
        super().__init__()

        self.features = features
        self.inchannels = inchannels
        self.outchannels = outchannels

        self.encoders = nn.ModuleList()
        self.decoders = nn.ModuleList()

        in_features = inchannels
        for idx, feature in enumerate(self.features):
            en = self.Encoder(in_features, feature)
            self.encoders.append(en)

            decoder_out = self.features[0] if idx == 0 else in_features

            de = self.Decoder(2*feature, decoder_out, final=(idx == 0))
            self.decoders.append(de)

            in_features = feature

        self.decoders = self.decoders[::-1]

        self.bottleneckLSTM = nn.LSTM(
                input_size=self.features[-1],
                hidden_size=self.features[-1] // 2,
                num_layers=4,
                batch_first=True,
                bidirectional=True,
                dropout=0.2)

        self.final = nn.Conv1d(self.features[0],outchannels,kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def Encoder(self,in_channels,out_channels):
        return nn.Sequential(
            nn.Conv1d(in_channels,in_channels,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.Conv1d(in_channels,2*in_channels,kernel_size=1),
            nn.GLU(dim=1),
            nn.Conv1d(in_channels,out_channels,kernel_size=8,stride=4,padding=1),
            nn.GroupNorm(8,out_channels),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

    def Decoder(self, in_channels, out_channels, final=False):

        layers = [
            nn.Conv1d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(in_channels, 2 * in_channels, kernel_size=1),
            nn.GLU(dim=1),
            nn.ConvTranspose1d(
                in_channels, out_channels,
                kernel_size=8, stride=4, padding=1
            ),
            nn.GroupNorm(8,out_channels)
        ]

        if not final:
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.2))

        return nn.Sequential(*layers)

    def forward(self,mixture):

        x = mixture
        skip = []

        for en in self.encoders:
            x = en(x)
            skip.append(x)

        x = x.transpose(1, 2)
        x, _ = self.bottleneckLSTM(x)
        x = x.transpose(1, 2)

        skip = skip[::-1]

        for i, de in enumerate(self.decoders):
            diff = skip[i].shape[-1] - x.shape[-1]
            
            if diff > 0:
                x = F.pad(x, (0, diff))
            elif diff < 0:
                x = x[..., :diff]
                
            x = torch.cat((x, skip[i]), dim=1)
            x = de(x)

        diff = mixture.shape[-1] - x.shape[-1]

        if diff > 0:
            x = F.pad(x, (0, diff))
        elif diff < 0:
            x = x[..., :mixture.shape[-1]]

        return self.final(x)

In [84]:
model = Demucs(inchannels=1, outchannels=2)
state_dict = torch.load("../models/demucs_vocal_inst_sep_last.pt", map_location=device)
model.load_state_dict(state_dict)
model = model.to(device)

In [85]:
SR = 44100  # sampling rate : number of numerical values per second of audio
CHUNK_DURATION = 5
CHUNK_SIZE = SR * CHUNK_DURATION  # total number of samples in a chunk
N_FFT = 2048  # number of samples in each FFT window
HOP_LENGTH = 512  # number of samples between successive frames
MODEL_INPUT_SIZE = 205008

In [86]:
def load_audio(path):
    audio, sr = librosa.load(path, sr=SR, mono=True)
    audio = np.array(audio)
    return audio

In [87]:
def get_chunks(signal):
    return [
        signal[i:i + CHUNK_SIZE]
        for i in range(0, len(signal), CHUNK_SIZE)
    ]

In [88]:
def inference_pipeline(path, model):

    mixture = load_audio(path)
    mixture_chunks = get_chunks(mixture)

    vocals = []
    instruments = []

    model.eval()

    with torch.inference_mode():

        for chunk in mixture_chunks:

            original_length = len(chunk)

            # Padding
            if original_length < MODEL_INPUT_SIZE:
                chunk = np.pad(chunk,(0, MODEL_INPUT_SIZE - original_length))

            waveform = torch.from_numpy(chunk).float()
            waveform = waveform.unsqueeze(0).unsqueeze(0).to(device)

            prediction = model(waveform)

            vocal = prediction[0, 0, :original_length]
            instrumental = prediction[0, 1, :original_length]

            vocals.append(vocal.cpu().numpy())
            instruments.append(instrumental.cpu().numpy())

    vocals = np.concatenate(vocals)
    instruments = np.concatenate(instruments)

    return vocals, instruments

In [89]:
test_files = [
    ("../test/test_audio.mp3", "../test/vocals.wav", "../test/instruments.wav"),
    ("../test/test_audio2.mp3", "../test/vocals2.wav", "../test/instruments2.wav"),
    ("../test/test_audio3.mp3", "../test/vocals3.wav", "../test/instruments3.wav")
]

In [90]:
for input_path, vocal_path, instrumental_path in test_files:

    vocals, instruments = inference_pipeline(input_path, model)

    sf.write(vocal_path, vocals, SR)
    sf.write(instrumental_path, instruments, SR)

    print(f"Processed: {input_path}")

Processed: ../test/test_audio.mp3
Processed: ../test/test_audio2.mp3
Processed: ../test/test_audio3.mp3
